# Fetal Ultrasound Scan Plane Quality Classifier (Phase 1)

**Question this notebook answers:** is a fetal ultrasound image one of the recognised standard views, or is it an "Other" image that is not a usable standard plane?

This is a scan quality check, not a diagnostic tool. The model only judges whether the right kind of view was captured.

**Data:** FETAL_PLANES_DB (public, Zenodo). 12,400 images with a plane label for each.

**How to run:** Runtime menu, Change runtime type, choose T4 GPU, then run every cell from top to bottom. If Colab disconnects, reconnect and rerun from the top. The download is saved in your Google Drive so it is only done once.

## Step 1. Check that the GPU is switched on
A GPU trains the model in minutes instead of hours. The next cell should print `True`.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: True


## Step 2. Download the dataset into Google Drive (once only)
Colab forgets everything when it disconnects, but Google Drive does not. We save the zip file in Drive. The `wget -c` option means that if the connection drops, rerunning this cell resumes the download instead of starting again. It needs about 2 GB of free Drive space.

In [7]:
!wget -c -O /content/fetal_planes.zip "https://zenodo.org/records/3904280/files/FETAL_PLANES_ZENODO.zip"
!ls -lh /content/fetal_planes.zip

--2026-10-14 08:51:29--  https://zenodo.org/records/3904280/files/FETAL_PLANES_ZENODO.zip
Resolving zenodo.org (zenodo.org)... 137.138.52.235, 137.138.153.219, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2088522169 (1.9G) [application/octet-stream]
Saving to: ‘/content/fetal_planes.zip’

/content/fetal_plan  47%[========>           ] 949.19M   452KB/s    in 30m 58s 

2026-10-14 09:23:15 (523 KB/s) - Connection closed at byte 995300084. Retrying.

--2026-10-14 09:23:16--  (try: 2)  https://zenodo.org/records/3904280/files/FETAL_PLANES_ZENODO.zip
Connecting to zenodo.org (zenodo.org)|137.138.52.235|:443... connected.
HTTP request sent, awaiting response... 206 PARTIAL_CONTENT
Length: 2088522169 (1.9G), 1093222085 (1.0G) remaining [application/octet-stream]
Saving to: ‘/content/fetal_planes.zip’

/content/fetal_plan 100%[+++++++++==========>]   1.94G   492KB/s    in 29m 50s 

2026-10-14 

## Step 3. Unzip onto Colab's fast local disk
The size printed above should be about 1.9G. If it is smaller, rerun the previous cell. Then this cell unzips the images. It takes a few minutes and prints nothing until it finishes.

## Step 4. Load the labels and create the yes or no label
The labels file uses semicolons instead of commas, so we tell pandas with `sep=';'`.

The `Plane` column has six values: Other, Maternal cervix, Fetal abdomen, Fetal brain, Fetal femur, Fetal thorax. We turn this into one yes or no column called `is_standard`, which is 1 for any real standard plane and 0 for "Other".

The dataset authors already mark which images are for training and which are for testing (the `Train` column). We keep their test set untouched until the very end. From the training images we set aside 20 percent for checking progress, and we split by **patient**, so that images from the same patient never appear on both sides. If they did, the model could look good just by recognising individual patients, and the score would be misleadingly high.

In [10]:
!ls -lh /content/fetal_planes.zip
!unzip -q -o /content/fetal_planes.zip -d /content/fetal_data
!ls /content/fetal_data

-rw-r--r-- 1 root root 2.0G Oct 14 09:53 /content/fetal_planes.zip
FETAL_PLANES_DB_data.csv  FETAL_PLANES_DB_data.xlsx  Images  README.md


In [13]:
import pandas as pd, numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

IMG_DIR = '/content/fetal_data/Images'
df = pd.read_csv('/content/fetal_data/FETAL_PLANES_DB_data.csv', sep=';')
df['is_standard'] = (df['Plane'] != 'Other').astype(int)

print(df['Plane'].value_counts())
print()
print('Standard vs Other:')
print(df['is_standard'].value_counts())
print('Baseline accuracy if a model always guessed standard:', round(df['is_standard'].mean(), 3))

train_all = df[df['Train '] == 1]
test_df = df[df['Train '] == 0].reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, val_idx = next(gss.split(train_all, groups=train_all['Patient_num']))
train_df = train_all.iloc[tr_idx].reset_index(drop=True)
val_df = train_all.iloc[val_idx].reset_index(drop=True)
print()
print('train / validation / test sizes:', len(train_df), len(val_df), len(test_df))

Plane
Other              4213
Fetal brain        3092
Fetal thorax       1718
Maternal cervix    1626
Fetal femur        1040
Fetal abdomen       711
Name: count, dtype: int64

Standard vs Other:
is_standard
1    8187
0    4213
Name: count, dtype: int64
Baseline accuracy if a model always guessed standard: 0.66

train / validation / test sizes: 5823 1306 5271


## Step 5. Prepare the images
Every image is resized to 224 by 224 pixels and normalised the way the pretrained model expects. For the training images only, we add small random rotations and brightness changes. This is called data augmentation. It stops the model from memorising exact images and makes it cope better with real scans that vary in angle and contrast.

In [16]:
class FetalDS(Dataset):
    def __init__(self, frame, tfm):
        self.frame, self.tfm = frame, tfm
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img = Image.open(f"{IMG_DIR}/{row['Image_name']}.png").convert('RGB')
        return self.tfm(img), int(row['is_standard'])

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.RandomRotation(10),
                                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                                transforms.ToTensor(), norm])
eval_tfm = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), norm])

train_dl = DataLoader(FetalDS(train_df, train_tfm), batch_size=64, shuffle=True, num_workers=2)
val_dl = DataLoader(FetalDS(val_df, eval_tfm), batch_size=64, num_workers=2)
test_dl = DataLoader(FetalDS(test_df, eval_tfm), batch_size=64, num_workers=2)
print('data loaders ready')

data loaders ready


## Step 6. Build and train the model
We use ResNet18, a standard image model that was already trained on millions of everyday photos. We keep what it has learned about edges and shapes and replace only its final layer with a two way decision (standard or other). This is called transfer learning, and it is why a small project can still reach strong results.

Training runs for 5 passes over the data (epochs). After each pass we measure accuracy on the validation images and keep the best version of the model.

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using', device)

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.0001)
best = 0

for epoch in range(5):
    model.train()
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    acc = correct / len(val_df)
    print(f'epoch {epoch + 1}  validation accuracy {acc:.3f}')
    if acc > best:
        best = acc
        torch.save(model.state_dict(), '/content/best_model.pt')

Using cuda
epoch 1  validation accuracy 0.947
epoch 2  validation accuracy 0.950
epoch 3  validation accuracy 0.947
epoch 4  validation accuracy 0.947
epoch 5  validation accuracy 0.943


## Step 7. Final test on images the model has never seen
This is the number that counts. The test images were never used for training or for choosing the best model.

How to read the results:
* **Recall for Standard plane**: of all genuine standard views, the share the model correctly accepted.
* **Recall for Other**: of all non standard images, the share the model correctly flagged. In a quality check tool this matters a lot, because a missed bad image is the costly mistake.
* **Confusion matrix**: rows are the true class, columns are the prediction. Top left and bottom right are correct. The other two cells are the mistakes.
* **AUC**: 1.0 is perfect separation, 0.5 is a coin flip.

In [19]:
model.load_state_dict(torch.load('/content//best_model.pt'))
model.eval()
probs, labels = [], []
with torch.no_grad():
    for x, y in test_dl:
        p = torch.softmax(model(x.to(device)), dim=1)[:, 1].cpu().numpy()
        probs.extend(p)
        labels.extend(y.numpy())

probs, labels = np.array(probs), np.array(labels)
preds = (probs >= 0.5).astype(int)
print(classification_report(labels, preds, target_names=['Other', 'Standard plane']))
print('Confusion matrix (rows true, columns predicted):')
print(confusion_matrix(labels, preds))
print('AUC:', round(roc_auc_score(labels, probs), 4))

                precision    recall  f1-score   support

         Other       0.92      0.79      0.85      1612
Standard plane       0.91      0.97      0.94      3659

      accuracy                           0.91      5271
     macro avg       0.91      0.88      0.89      5271
  weighted avg       0.91      0.91      0.91      5271

Confusion matrix (rows true, columns predicted):
[[1268  344]
 [ 113 3546]]
AUC: 0.9666


## Step 8. Where does it go wrong?
This checks accuracy separately for each of the six original plane types. A model can look good overall while failing on one specific view, and this table shows that honestly.

In [20]:
test_df['pred_standard'] = preds
test_df['correct'] = (test_df['pred_standard'] == test_df['is_standard']).astype(int)
print(test_df.groupby('Plane')['correct'].agg(['mean', 'count']).round(3))

                  mean  count
Plane                        
Fetal abdomen    0.958    358
Fetal brain      0.998   1472
Fetal femur      0.922    524
Fetal thorax     0.936    660
Maternal cervix  0.981    645
Other            0.787   1612


## Limitations to state honestly
* "Other" means the image is not one of the target views. It is a stand in for inadequate, not a clinical quality grade.
* The images come from a limited set of operators and ultrasound machines, so results may not carry over to other hospitals or devices.
* This model has not been clinically validated and is not for patient care.

## Next: Phase 2
Run the trained classifier frame by frame over a recorded ultrasound video and show live style feedback on whether the correct plane is being captured.